(what-is-ecological-economics)=
# What Is Ecological Economics?

Open most economics textbooks and "the economy" is treated as the whole picture: households, firms and markets trading goods and services, with nature showing up mostly as a source of raw materials, or as an "externality" you price in once something has already gone wrong.

Ecological economics starts somewhere else. The economy isn't the whole system — it's a *subsystem*, sitting inside a biosphere that is finite and doesn't grow. Everything the economy makes or consumes has to come from somewhere (energy and material stocks) and go somewhere (waste heat, emissions, degraded materials), and both of those somewheres are the same biosphere. Take that seriously and the central question changes. It's no longer just "how do we allocate scarce resources efficiently," but also "how big can this subsystem get before it runs into the size of the system holding it?"

To put a number on that: the [Python intro chapter](../intro_python/calculator.ipynb) already has you working with roughly **592 EJ** of primary energy consumed per year by about **8.2 billion people** — call it **72 GJ per person per year**. That whole flow, everything needed to run farms, factories, transport and homes for the entire human economy, is what "subsystem" means here. It has to come from somewhere, and it has to go somewhere, and both of those somewheres are the biosphere.

## A biophysical starting point

This isn't really an economics idea to begin with — it's closer to physics. A number of the people who founded the field weren't trained as economists at all; they came from mathematics and physics.

### Georgescu-Roegen: entropy is a one-way street

[Nicholas Georgescu-Roegen](https://en.wikipedia.org/wiki/Nicholas_Georgescu-Roegen) made the case that production doesn't get to ignore the laws of thermodynamics. Specifically the second one: entropy. Material and energy transformations only run one way — you don't get a closed loop.

Here's a toy version of that. Start with 100 units of a usable resource. Every cycle you use some of it, recycling recovers a fraction of what you used, and the rest becomes waste you can't use again. Push the recycling slider all the way to 95% and watch what still happens: the usable stock keeps shrinking, cycle after cycle. Recycling slows things down a lot. It just never gets you back to where you started.

In [1]:
import plotly.graph_objects as go
from IPython.display import HTML


def entropy_depletion(stock0=100.0, use_rate=0.10, recycling_efficiency=0.0, cycles=30):
    stock = stock0
    waste = 0.0
    stocks = [stock]
    wastes = [waste]
    for _ in range(cycles):
        used = stock * use_rate
        recovered = used * recycling_efficiency
        lost = used - recovered
        stock = stock - lost
        waste += lost
        stocks.append(stock)
        wastes.append(waste)
    return stocks, wastes


recycling_levels = [0.0, 0.25, 0.5, 0.75, 0.95]
cycles_axis = list(range(31))

fig = go.Figure()
for i, eff in enumerate(recycling_levels):
    stocks, wastes = entropy_depletion(recycling_efficiency=eff)
    fig.add_trace(go.Scatter(x=cycles_axis, y=stocks, mode="lines", name="Usable stock",
                              line=dict(color="#2E7D32", width=3), visible=(i == 0)))
    fig.add_trace(go.Scatter(x=cycles_axis, y=wastes, mode="lines", name="Cumulative waste",
                              line=dict(color="#B71C1C", width=3, dash="dash"), visible=(i == 0)))

steps = []
for i, eff in enumerate(recycling_levels):
    visible = [False] * (2 * len(recycling_levels))
    visible[2 * i] = True
    visible[2 * i + 1] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{eff:.0%}"))

fig.update_layout(
    sliders=[dict(active=0, currentvalue={"prefix": "Recycling efficiency: "}, steps=steps)],
    title="Even near-perfect recycling only slows entropy increase \u2014 it never reverses it",
    xaxis_title="Cycle (e.g. years)",
    yaxis_title="Units (usable stock vs. cumulative waste)",
    height=450,
)
HTML(fig.to_html(include_plotlyjs="cdn"))

Even at 95% recycling, the stock after 30 cycles is still lower than where it started. At 0% recycling it's down to about 4% of the original. So recycling clearly matters — it just buys time rather than closing the loop.

### Ayres: economies have a metabolism

[Robert Ayres](https://en.wikipedia.org/wiki/Robert_Ayres_(scientist)) took this further with the idea of *industrial metabolism*: economies, like organisms, need a continuous flow of energy and materials through them just to keep functioning. No throughput, no activity.

The chart below is real 2023 data across countries (the same file the [static scaling chapter](../order_of_magnitude_economics_and_scaling/static_scale.ipynb) uses later) — energy use per capita against GDP per capita, on a log-log scale. Move the slider through different target GDP levels and see what energy throughput the fitted trend expects a country to need at that income.

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML

df = pd.read_csv("../../teaching_data_sets/data2_energygdp_owid2023.csv", sep=";")
df = df.dropna(subset=["energy_pc", "gdp_pc"])
df = df[(df["energy_pc"] > 0) & (df["gdp_pc"] > 0)]

log_gdp = np.log10(df["gdp_pc"])
log_energy = np.log10(df["energy_pc"])
slope, intercept = np.polyfit(log_gdp, log_energy, 1)


def predict_energy(gdp_pc):
    return 10 ** (slope * np.log10(gdp_pc) + intercept)


target_gdps = [2000, 8000, 20000, 50000, 100000]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df["gdp_pc"], y=df["energy_pc"], mode="markers",
    marker=dict(size=7, color="#1565C0", opacity=0.6),
    name="Countries (2023)",
))
x_line = np.logspace(np.log10(df["gdp_pc"].min()), np.log10(df["gdp_pc"].max()), 100)
fig.add_trace(go.Scatter(
    x=x_line, y=predict_energy(x_line), mode="lines",
    line=dict(color="#EF6C00", width=2), name="Fitted trend",
))

for i, target in enumerate(target_gdps):
    predicted = predict_energy(target)
    fig.add_trace(go.Scatter(
        x=[target, target], y=[df["energy_pc"].min(), predicted],
        mode="lines", line=dict(color="#6A1B9A", width=2, dash="dot"),
        visible=(i == 0), name="Target GDP per capita", showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=[target], y=[predicted], mode="markers+text",
        marker=dict(size=12, color="#6A1B9A", symbol="diamond"),
        text=[f"{predicted:,.0f} kWh/person"], textposition="top center",
        visible=(i == 0), name="Predicted energy throughput", showlegend=False,
    ))

n_base = 2
steps = []
for i, target in enumerate(target_gdps):
    visible = [True, True] + [False] * (2 * len(target_gdps))
    visible[n_base + 2 * i] = True
    visible[n_base + 2 * i + 1] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"${target:,}"))

fig.update_layout(
    sliders=[dict(active=0, currentvalue={"prefix": "Target GDP per capita: "}, steps=steps)],
    xaxis=dict(type="log", title="GDP per capita ($)"),
    yaxis=dict(type="log", title="Energy consumption per capita (kWh)"),
    title="Sustaining a given GDP requires a matching energy ‘metabolism’ (2023 cross-country data)",
    height=500,
)
HTML(fig.to_html(include_plotlyjs="cdn"))

The fitted slope comes out around 1.2, so energy use per capita actually rises a bit *faster* than GDP per capita across these 189 countries, not just in step with it. Nobody in this dataset is running a high-GDP lifestyle on low energy throughput.

### Daly: growth economy vs. steady-state economy

[Herman Daly](https://en.wikipedia.org/wiki/Herman_Daly) gave this a name: the **steady-state economy**. Physical throughput stays flat, capped by ecological limits, while everything qualitative — knowledge, technology, wellbeing — is still free to keep improving.

The chart plots real global energy-consumption-per-capita data from 1965 to 2024 against a few hypothetical steady-state caps. Pick a scenario from the slider and see how different a capped path looks next to what actually happened.

In [3]:
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML

df = pd.read_csv("../../teaching_data_sets/data1_energygdp_owid.csv", on_bad_lines="skip")
world = df[df["Entity"] == "World"].sort_values("Year")
years = world["Year"].tolist()
energy = world["Per capita energy consumption"].tolist()

steady_state_options = [
    {"level": energy[years.index(1995)], "start_year": 1995, "label": "Cap at 1995 level (~17,600 kWh)"},
    {"level": 16000, "start_year": 1995, "label": "Cap at 16,000 kWh, starting 1995"},
    {"level": 16000, "start_year": 2010, "label": "Cap at 16,000 kWh, starting 2010"},
    {"level": 18000, "start_year": 2010, "label": "Cap at 18,000 kWh, starting 2010"},
]


def steady_state_path(level, start_year):
    return [e if y < start_year else level for y, e in zip(years, energy)]


fig = go.Figure()
fig.add_trace(go.Scatter(x=years, y=energy, mode="lines", name="Actual growth trajectory",
                          line=dict(color="#B71C1C", width=3)))

for i, opt in enumerate(steady_state_options):
    path = steady_state_path(opt["level"], opt["start_year"])
    fig.add_trace(go.Scatter(x=years, y=path, mode="lines", name="Steady-state proposal",
                              line=dict(color="#2E7D32", width=3, dash="dash"),
                              visible=(i == 0), showlegend=(i == 0)))

steps = []
for i, opt in enumerate(steady_state_options):
    visible = [True] + [False] * len(steady_state_options)
    visible[1 + i] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=opt["label"]))

fig.update_layout(
    sliders=[dict(active=0, currentvalue={"prefix": "Steady-state scenario: "}, steps=steps)],
    xaxis_title="Year", yaxis_title="World primary energy consumption per capita (kWh)",
    title="Growth economy vs. Daly’s steady-state proposal (real global data, 1965-2024)",
    height=450,
)
HTML(fig.to_html(include_plotlyjs="cdn"))

None of these scenarios claim the world actually did this — they're counterfactuals, not history. The point is just structural: a steady-state economy means capping physical throughput, not freezing progress or wellbeing.

### Boulding: cowboy economy vs. spaceman economy

[Kenneth Boulding](https://en.wikipedia.org/wiki/Kenneth_Boulding) put this memorably: the old "cowboy economy" (endless frontier, someone else deals with the mess) versus the "spaceman economy" (a closed ship, where both what comes in and what gets thrown out are limited).

Both economies below start with the same 1,000-unit stock. Cowboy pulls out a fixed amount every year regardless of what's left — a stock that never comes back. Spaceman pulls out the same fixed amount too, but from a stock that regenerates 5% a year, up to its original size. Renewable versus non-renewable, basically. Push the extraction rate slider up and see what happens to each.

In [4]:
import plotly.graph_objects as go
from IPython.display import HTML


def simulate_boulding(stock0=1000.0, regen_rate=0.05, extraction_rate=0.05, years=40):
    cowboy = stock0
    spaceman = stock0
    cowboy_path = [cowboy]
    spaceman_path = [spaceman]
    fixed_harvest = extraction_rate * stock0
    for _ in range(years):
        cowboy = max(0.0, cowboy - fixed_harvest)
        harvest_s = min(extraction_rate * stock0, spaceman)
        spaceman = min(stock0, (spaceman - harvest_s) * (1 + regen_rate))
        cowboy_path.append(cowboy)
        spaceman_path.append(spaceman)
    return cowboy_path, spaceman_path


extraction_levels = [0.02, 0.04, 0.06, 0.08, 0.10]
years_axis = list(range(41))

fig = go.Figure()
for i, rate in enumerate(extraction_levels):
    cowboy_path, spaceman_path = simulate_boulding(extraction_rate=rate)
    fig.add_trace(go.Scatter(x=years_axis, y=cowboy_path, mode="lines", name="Cowboy economy",
                              line=dict(color="#B71C1C", width=3), visible=(i == 0)))
    fig.add_trace(go.Scatter(x=years_axis, y=spaceman_path, mode="lines", name="Spaceman economy",
                              line=dict(color="#2E7D32", width=3), visible=(i == 0)))

steps = []
for i, rate in enumerate(extraction_levels):
    visible = [False] * (2 * len(extraction_levels))
    visible[2 * i] = True
    visible[2 * i + 1] = True
    steps.append(dict(method="update", args=[{"visible": visible}], label=f"{rate:.0%} of stock/year"))

fig.update_layout(
    sliders=[dict(active=0, currentvalue={"prefix": "Extraction rate: "}, steps=steps)],
    xaxis_title="Year", yaxis_title="Resource stock remaining (units)",
    title="Cowboy vs. spaceman economy \u2014 extraction rate vs. regeneration capacity (5%/year)",
    height=450,
)
HTML(fig.to_html(include_plotlyjs="cdn"))

At low extraction rates, spaceman levels off while cowboy still grinds its stock down to zero. But push extraction past the 5%/year regeneration rate and spaceman collapses too. Having *some* limit isn't enough — it has to be the right one.

## How this differs from environmental economics

People mix up ecological economics with *environmental economics* a lot, and honestly the two overlap plenty in practice. But they start from different places.

- **Environmental economics** mostly stays inside the standard neoclassical toolkit. Nature's services get a price tag (or a shadow price), and environmental problems are treated as market failures to correct. Underneath that is usually **weak sustainability**: natural capital can, in principle, be swapped for produced capital, so long as the total doesn't shrink.
- **Ecological economics** tends to lean toward **strong sustainability** instead. Some forms of natural capital — a stable climate, a functioning ecosystem, biodiversity — aren't things you can substitute your way out of. No amount of built capital buys them back.

### A worked example: draining a wetland

Picture a country draining a wetland to build a factory district. The wetland was quietly worth something like $50 million a year in flood protection alone, plus a bunch of harder-to-price stuff: habitat, water filtration, carbon storage. The factories built where it used to be are worth $200 million.

- **Weak sustainability says:** fine. Total capital went up — $50M/year of natural capital for $200M of produced capital — and as long as some of that gets reinvested into things like levees to cover the lost flood protection, nothing's really been lost.
- **Strong sustainability says:** not so fast. No levee does everything that wetland was doing at once — flood buffering, habitat, filtration, carbon storage, all together. Some of what was there is gone for good, whatever the balance sheet says.

In line with the [Preface](../preface.md), this book mostly stays descriptive and quantitative rather than picking a side here. But you need to know this distinction exists before the rest of the field makes much sense.

## What ecological economists actually do

Day to day, this mostly means building quantitative tools that treat the economy and the environment as one system, not two:

- **Scaling and order-of-magnitude reasoning.** GDP, energy use, emissions — these things span huge ranges across countries and time, and you need an intuition for that. The {ref}`next chapter <order_of_magnitude_economics_and_scaling>` builds exactly this, and you already saw it above: GDP per capita in the Ayres chart ranges from roughly \$1,000 to \$130,000, about two orders of magnitude.
- **Biophysical accounting.** Energy and material flow analysis, footprint accounting, input-output analysis — tracking physical flows the same way you'd track money. That's what the Ayres and Daly charts above are actually doing.
- **Modelling coupled systems.** Agent-based models, system dynamics — letting population, resources, technology and output evolve together instead of holding the environment fixed in the background. The Boulding simulation above is a taste of this (a full chapter on these techniques is coming later in the book).

This is the toolkit the rest of the book builds, chapter by chapter, with Python doing the heavy lifting (see {ref}`intro-python`).

:::{tip}
**What you'll learn in this chapter**
- What distinguishes ecological economics from mainstream (neoclassical) and environmental economics
- The biophysical/thermodynamic starting point of the field (Georgescu-Roegen, Ayres, Daly, Boulding), made concrete with interactive models you can play with
- The difference between weak and strong sustainability, with a worked example
- What kinds of quantitative tools ecological economists use, and how they map onto the rest of this book
:::